In [2]:
# init
import requests
import pickle
import os
from pathlib import Path
from importlib import reload
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsPandas.helpers as tph
import toolsSync.main as tsm
import pandas as pd
import copy
import re
from dotenv import load_dotenv
import boto3

import shutil

def pckgs_reload():
    reload(tgm)
    reload(tph)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
SCRAPE_DIR = DATA_DIR / 'raw/osm countries queries'
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
TEST_FIRST_LEVEL_DIR = TESTS_DIR / 'osm first level test'
TEST_DUPLICATES_DIR = TESTS_DIR / 'osm duplicates test'

logger = tgl.initiate_logger('logger')

ModuleNotFoundError: No module named 'requests'

In [53]:
# init
load_dotenv()
bucket_name = os.environ["B2_BUCKET_NAME"]
session = boto3.session.Session()
s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

config = {'root':ROOT, 's3':s3, 'logger':logger}

## Review state of tasks

In [1]:
process_state = tgm.load(DATA_DIR / 'process_state.json')
print(len(process_state))

task_state = {
    'scrape':{'dir':'data/raw/osm countries queries', 'required_task':'scrape'},
    'clean':{'dir':'data/cleaned', 'required_task':'scrape'},
    'test_basic':{'dir':'data/tests results/osm basic test','required_task':'clean'},
    'test_first_level':{'dir':'data/tests results/osm first level test','required_task':'clean'},
    'test_duplicates':{'dir':'data/tests results/osm duplicates test','required_task':'clean'}
}

for task in task_state.keys():
    task_path = ROOT / task_state[task]['dir']
    ok = [c for c, val in process_state.items() if (val[task]['status'] == 'ok')]
    pending = [c for c, val in process_state.items() if (val[task]['status'] == 'pending')]
    error = [c for c, val in process_state.items() if (val[task]['status'] == 'error')]
    missing = [c for c, val in process_state.items() if (val[task]['status'] == 'missing')]
    countries_dirs = [dir.relative_to(task_path).name for dir in task_path.glob('*') if dir.is_dir()]
    
    task_state[task]['ok'] = ok
    task_state[task]['pending'] = pending
    task_state[task]['error'] = error
    task_state[task]['missing'] = missing
    previous_task_ok = task_state[task_state[task]['required_task']]['ok']
    to_process = tgm.complement(tgm.complement(previous_task_ok, ok), missing)
    task_state[task]['to_process'] = to_process
    task_state[task]['local_dirs'] = countries_dirs
    task_state[task]['ok_wihout_local_dirs'] = tgm.complement(ok, countries_dirs)
    previous_task_local_dirs = task_state[task_state[task]['required_task']]['local_dirs']
    task_state[task]['to_process_wihout_required_local_dirs'] = tgm.complement(to_process, previous_task_local_dirs)

# convert to length
task_state_resume = copy.deepcopy(task_state)
for task,val in task_state.items():
    for k,v in task_state_resume[task].items():
        if k in ['dir', 'required_task']: continue
        task_state_resume[task][k] = len(v)

    state_resume_df = pd.DataFrame(task_state_resume)
    state_df = pd.DataFrame(task_state)

NameError: name 'tgm' is not defined

In [7]:
state_resume_df.T

,dir,required_task,ok,pending,error,missing,to_process,local_dirs,ok_wihout_local_dirs,to_process_wihout_required_local_dirs
scrape,data/raw/osm countries queries,scrape,214,0,0,0,0,95,119,0
clean,data/cleaned,scrape,214,0,0,0,0,95,119,0
test_basic,data/tests results/osm basic test,clean,214,0,0,0,0,84,130,0
test_first_level,data/tests results/osm first level test,clean,69,131,0,14,131,66,3,119
test_duplicates,data/tests results/osm duplicates test,clean,33,131,0,50,131,32,1,119


In [59]:
state_df['scrape']['error']

['Jamaica']

In [4]:
tgm.complement(task_state['clean']['ok'],task_state['scrape']['ok'])

['Bahrain', 'Botswana']

# Fix process state

## * initialize state

In [90]:
osmMetaCountrDict = tgm.load(DATA_DIR / 'osmMetaCountrDict.json')
print(len(osmMetaCountrDict))

241


In [91]:
process_state = {}
for country in osmMetaCountrDict.keys():
    process_state[country] = {
        "scrape": {
            "status": "pending",
            "type": 'normal',
            "last_run": None,
            "error": None
        },
        "clean": {
            "status": "pending",
            "last_run": None,
            "error": None
        },
        "test_basic": {
            "status": "pending",
            "last_run": None,
            "error": None
        },
        "test_first_level": {
            "status": "pending",
            "last_run": None,
            "error": None
        },
        "test_duplicates": {
            "status": "pending",
            "last_run": None,
            "error": None
        },
        "fix": {
            "status": "pending",
            "last_run": None,
            "error": None
        }
    }
print(len(process_state))

241


In [154]:
tgm.dump(DATA_DIR / 'process_state.json', process_state)

## * check tasks state

In [203]:
scrape_ok = df.query("task == 'scrape' and status == 'ok'")["country"]
print(len(scrape_ok))

98


In [161]:
df.query("country in @ok_scrape").df_peek()

,country,task,status,type,last_run,error,chunk_state
787,Afghanistan,clean,pending,NaN,None,None,NaN
791,Afghanistan,fix,pending,NaN,None,None,NaN
786,Afghanistan,scrape,ok,normal,2025-12-02T16:06:44,None,NaN
788,Afghanistan,test_basic,pending,NaN,None,None,NaN
790,Afghanistan,test_duplicates,pending,NaN,None,None,NaN
789,Afghanistan,test_first_level,pending,NaN,None,None,NaN
1399,AkrotiriAndDhekelia,clean,pending,NaN,None,None,NaN
1403,AkrotiriAndDhekelia,fix,pending,NaN,None,None,NaN
1398,AkrotiriAndDhekelia,scrape,ok,normal,2025-12-02T16:06:44,None,NaN
1400,AkrotiriAndDhekelia,test_basic,pending,NaN,None,None,NaN


In [204]:
clean_pending = df.query("country in @ok_scrape").query("task == 'clean' and status == 'pending'")["country"]
print(len(clean_pending))

26


# Test duplicates

## * recompute duplicates test state

In [254]:
dups_test_state = tgm.load(DATA_DIR / 'dups_test_state.json')
print(f"Countries dups state: {len(dups_test_state)}")

processed_id_triplets = [ids for res in dups_test_state.values() for ids in res['processed']]
print(f"Processed id triplets: {len(processed_id_triplets)}")

Countries dups state: 214
Processed id triplets: 1819


In [249]:
dups_test_state = {}
for c,res in process_state.items():
    dups_test_state[c] = {'to_process':set(), 'processed':set(), 'failed':set(), "next_index":0}
print(len(dups_test_state))

214


In [250]:
dirs = [dir for dir in TEST_DUPLICATES_DIR.glob("*") if dir.is_dir()]
dups_test_res = {}
country_last_index = {}
for dir in dirs:
    if not dups_test_res.get(dir.name):
        dups_test_res[dir.name] = {}
    files = [f for f in dir.glob(f"*") if f.is_file()]
    indexes = sorted([int(re.search(r'_dups_test_res_(\d+)\.pkl', str(f)).group(1)) for f in files])
    country_last_index[dir.name] = indexes[-1]
    if len(files)>1: print(dir.name)
    for f in files:
        data = tgm.load(f)
        dups_test_res[dir.name].update(data)

print(f'Results per country found: {len(dups_test_res)}')

Results per country found: 30


In [251]:
for country, res in dups_test_res.items():
    for id_triplet, rel_res in res.items():
        statuses = [v['status'] for k,v in rel_res.items()]
        if 'error' in statuses:
            dups_test_state[country]['failed'].add(id_triplet)
        # 'ok' and 'missing' are considered processed
        else:
            dups_test_state[country]['processed'].add(id_triplet)
    dups_test_state[country]['next_index'] = country_last_index[country]+ 1
        
print(len(dups_test_state))

214


In [252]:
processed_id_triplets = [ids for res in dups_test_state.values() for ids in res['processed']]
print(f"Processed id triplets: {len(processed_id_triplets)}")

Processed id triplets: 1819


In [253]:
tgm.dump(DATA_DIR / 'dups_test_state.json', dups_test_state)

## * update process state

In [41]:
dups_test_state = tgm.load(DATA_DIR / 'dups_test_state.json')
print(len(dups_test_state))

process_state = tgm.load(DATA_DIR / 'process_state.json')
print(len(process_state))

30
214


In [40]:
failed = [c for c,v in dups_test_state.items() if len(v['failed']) > 0]
print(f"Countries with failed tests: {len(failed)}")
ok = [c for c,v in dups_test_state.items() if len(v['failed']) < 1]
print(f"Countries with ok tests: {len(ok)}")

Countries with failed tests: 0
Countries with ok tests: 30


In [42]:
for c in ok:
    tsm.update_process_state(process_state, c, 'test_duplicates', process_status="ok")

In [43]:
tgm.dump(DATA_DIR / 'process_state.json', process_state)

## * recompute dups_ids

In [2]:
cleaned_by_cntr = tgm.load_dirs(CLEANED_DIR)
print(f'Results per country found: {len(cleaned_by_cntr)}')

Results per country found: 86


In [4]:
df_all = pd.concat(cleaned_by_cntr.values(), ignore_index=True)
print(len(df_all))

id_triplets = df_all[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).to_list()
print(len(id_triplets))

198690
198690


In [5]:
dups_id = set(tgm.find_duplicates([t[0] for t in id_triplets]))
print(f"duplicates (id): {len(dups_id)}")

print(f"duplicates (id, parent_id): {len(set(tgm.find_duplicates([(t[0],t[1]) for t in id_triplets])))}")

print(f"duplicates (id, parent_id, country_id): {len(set(tgm.find_duplicates([(t[0],t[1],t[2]) for t in id_triplets])))}")


duplicates (id): 8026
duplicates (id, parent_id): 6560
duplicates (id, parent_id, country_id): 0


In [6]:
dups_id_tripets = [t for t in id_triplets if t[0] in dups_id]
print(f'Relations (rows) of those duplicates (id): {len(dups_id_tripets)}')

Relations (rows) of those duplicates (id): 17458


In [62]:
tgm.dump(TEST_DUPLICATES_DIR / 'dups_id.pkl', dups_id)

# Recompute ids from cleaned

In [25]:
cleaned_by_cntr = tgm.load_dirs(CLEANED_DIR)
print(f'Results per country found: {len(cleaned_by_cntr)}')

Results per country found: 83


In [26]:
df_all = pd.concat(cleaned_by_cntr.values(), ignore_index=True)
print(len(df_all))

id_triplets = df_all[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).to_list()
print(len(id_triplets))

194834
194834


In [27]:
ids = [ele[0] for ele in id_triplets]
print(len(ids))
ids_nodup = list(set(ids))
print(len(ids_nodup))

194834
185411


In [28]:
tgm.dump(CLEANED_DIR / 'ids.pkl', ids)

In [29]:
tsm.upload_file_to_backblaze(CLEANED_DIR / 'ids.pkl', config)

[2026-03-19 18:22:41] [INFO] : Uploaded c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\ids.pkl to Backblaze successfully


{'status': 'ok', 'status_type': None, 'data': None}